# Pipeline de Unión de características de ciudades y datos socioeconómicos

Este *notebook* implementa integración de la información estructural disponible para cada ciudad con los registros socioeconómicos anuales de población, PIB y número de vehículos por cada 1000 habitantes.

Los dos conjuntos de datos presentan inicialmente una granularidad diferente:

- el dataset de ciudades contiene una única observación por ciudad;
- el dataset socioeconómico contiene una observación por ciudad y año.

Para poder combinar ambos conjuntos, la información socioeconómica se agregará previamente a nivel de ciudad. Para cada variable, se calculará su valor mediano durante el período disponible, obteniendo así una caracterización representativa de cada ciudad y reduciendo la influencia de posibles valores extremos o discontinuidades puntuales. Posteriormente, ambos conjuntos se unirán utilizando el nombre normalizado de la ciudad como clave.

El resultado será un nuevo *dataset* con una única observación por ciudad que incorporará tanto las características originales como las variables socioeconómicas agregadas. Este fichero se utilizará posteriormente como entrada para el análisis de clustering de ciudades.

In [213]:
# =============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# =============================================================================

from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Configuración de visualización
# -----------------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

In [ ]:
# =============================================================================
# 2. CONFIGURACIÓN DE RUTAS
# =============================================================================

# El notebook se encuentra en:
# scripts/Ingesta y Preprocesamiento de datos
#
# Se retroceden dos niveles hasta la raíz del proyecto TFM_github.
BASE_PATH = Path("..", "..")


# -----------------------------------------------------------------------------
# Carpetas de entrada
# -----------------------------------------------------------------------------

FOLDER_CIUDADES = (
    BASE_PATH
    / "datasets"
    / "archivo_ciudades"
)

FOLDER_SOCIOECONOMICOS = (
    BASE_PATH
    / "datasets"
    / "archivos_socioeconómicos"
)


# -----------------------------------------------------------------------------
# Carpeta de salida
# -----------------------------------------------------------------------------

FOLDER_OUTPUT = (
    BASE_PATH
    / "datasets"
    / "archivo_ciudades_socioeconomicos"
)

FOLDER_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)


# -----------------------------------------------------------------------------
# Ficheros
# -----------------------------------------------------------------------------

RUTA_CIUDADES = (
    FOLDER_CIUDADES
    / "ciudades.csv"
)

RUTA_SOCIOECONOMICO = (
    FOLDER_SOCIOECONOMICOS
    / "dataset_socioeconómico.csv"
)

RUTA_SALIDA = (
    FOLDER_OUTPUT
    / "dataset_ciudades_socioeconomico.csv"
)

## Funciones auxiliares

Para realizar correctamente la unión entre ambos conjuntos de datos, se define una función de normalización de texto.

Los nombres de una misma ciudad pueden presentar pequeñas diferencias entre fuentes, por ejemplo en el uso de mayúsculas, tildes o espacios. Si se realiza una unión directamente sobre estas cadenas, dichas diferencias pueden impedir que se produzca una correspondencia correcta.

La normalización transforma los nombres a una representación homogénea eliminando tildes, convirtiendo el texto a minúsculas y sustituyendo caracteres especiales por guiones bajos. También se define una función de lectura de CSV capaz de intentar automáticamente diferentes separadores.

In [215]:
# =============================================================================
# 3. FUNCIONES AUXILIARES
# =============================================================================

def normalizar_texto(texto):
    """
    Convierte una cadena a una representación homogénea para facilitar
    comparaciones entre nombres procedentes de distintas fuentes.
    """

    texto = str(texto).strip()

    # Separación de caracteres base y signos diacríticos.
    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    # Eliminación de tildes y otros signos diacríticos.
    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    texto = texto.lower()

    # Sustitución de caracteres no alfanuméricos por "_".
    texto = re.sub(
        r"[^a-z0-9]+",
        "_",
        texto
    )

    # Eliminación de separadores consecutivos.
    texto = re.sub(
        r"_+",
        "_",
        texto
    ).strip("_")

    return texto


def leer_csv_robusto(ruta):
    """
    Lee un fichero CSV intentando detectar automáticamente
    el separador utilizado.
    """

    try:

        return pd.read_csv(
            ruta,
            sep=None,
            engine="python",
            encoding="utf-8-sig"
        )

    except Exception:

        try:

            return pd.read_csv(
                ruta,
                sep=";",
                encoding="utf-8-sig"
            )

        except Exception:

            return pd.read_csv(
                ruta,
                sep=",",
                encoding="utf-8-sig"
            )

## Carga de los conjuntos de datos

Se cargan los dos conjuntos que serán integrados.

Antes de realizar cualquier transformación se comprueba su dimensión y estructura. Esta inspección permite verificar que el dataset de ciudades contiene una única observación por ciudad y que el socioeconómico mantiene la estructura esperada de registros anuales.

In [216]:
# =============================================================================
# 4. CARGA DE LOS DATASETS
# =============================================================================

df_ciudades = leer_csv_robusto(
    RUTA_CIUDADES
)

df_socioeconomico = leer_csv_robusto(
    RUTA_SOCIOECONOMICO
)


print("DATASET DE CIUDADES")
print("-" * 60)

print(
    f"Filas: {df_ciudades.shape[0]} | "
    f"Columnas: {df_ciudades.shape[1]}"
)

display(
    df_ciudades.head()
)


print("\nDATASET SOCIOECONÓMICO")
print("-" * 60)

print(
    f"Filas: {df_socioeconomico.shape[0]} | "
    f"Columnas: {df_socioeconomico.shape[1]}"
)

display(
    df_socioeconomico.head()
)

DATASET DE CIUDADES
------------------------------------------------------------
Filas: 12 | Columnas: 7


,Ciudad,Latitud,Longitud,Altitud (m),Superficie (km²),Distancia_Mar (km),TPI_Factor_Cuenca
0,Madrid,4.041.694,-370.33300,660.0,604.45,327.66,14.10
1,Barcelona,413.825,217.69400,30.0,101.35,2.60,-48.24
2,Bilbao,4.326.306,-2.93500,24.0,41.60,8.75,-150.72
3,Sevilla,3.738.861,-599.55600,17.0,141.42,53.66,-21.20
4,Valencia,39.47,-0.37639,23.0,134.65,3.99,-1.72



DATASET SOCIOECONÓMICO
------------------------------------------------------------
Filas: 144 | Columnas: 5


,Ciudad,Años,Población,PIB,Vehiculos_1000_hab
0,A Coruña,2013,245923,20600,"352,61"
1,A Coruña,2014,244810,20600,"354,22"
2,A Coruña,2015,243870,21800,"355,33"
3,A Coruña,2016,243978,22700,"357,16"
4,A Coruña,2017,244099,23300,"332,69"


## Comprobación de la estructura de los datos

Antes de realizar la unión se comprueban los nombres de las columnas y sus tipos de datos.

Esta revisión es especialmente importante para el dataset socioeconómico, ya que algunas variables numéricas pueden haberse leído como texto. Esto puede ocurrir, por ejemplo, cuando los valores decimales utilizan coma como separador decimal.

También se verifican los nombres de las ciudades presentes en ambos conjuntos para detectar posibles diferencias entre las dos fuentes.

In [217]:
# =============================================================================
# 5. INSPECCIÓN DE LA ESTRUCTURA
# =============================================================================

print("COLUMNAS DEL DATASET DE CIUDADES")
print("-" * 60)

display(
    pd.DataFrame({
        "columna": df_ciudades.columns,
        "tipo": df_ciudades.dtypes.astype(str).values
    })
)


print("\nCOLUMNAS DEL DATASET SOCIOECONÓMICO")
print("-" * 60)

display(
    pd.DataFrame({
        "columna": df_socioeconomico.columns,
        "tipo": df_socioeconomico.dtypes.astype(str).values
    })
)

COLUMNAS DEL DATASET DE CIUDADES
------------------------------------------------------------


,columna,tipo
0,Ciudad,object
1,Latitud,object
2,Longitud,float64
3,Altitud (m),float64
4,Superficie (km²),float64
5,Distancia_Mar (km),float64
6,TPI_Factor_Cuenca,float64



COLUMNAS DEL DATASET SOCIOECONÓMICO
------------------------------------------------------------


,columna,tipo
0,Ciudad,object
1,Años,int64
2,Población,int64
3,PIB,int64
4,Vehiculos_1000_hab,object


## Conversión de las variables socioeconómicas

Las variables que posteriormente se agregarán deben encontrarse en formato numérico.

Se convierten explícitamente:

- `Población`;
- `PIB`;
- `Vehiculos_1000_hab`;
- `Años`.

Para la variable de vehículos se sustituye previamente la coma decimal por un punto, evitando que valores como `452,61` sean interpretados como cadenas de texto.

Los valores que no puedan convertirse correctamente se transformarán en valores ausentes (`NaN`), lo que permite detectarlos posteriormente.

In [218]:
# =============================================================================
# 6. CONVERSIÓN DE VARIABLES SOCIOECONÓMICAS
# =============================================================================

df_socioeconomico["Población"] = pd.to_numeric(
    df_socioeconomico["Población"],
    errors="coerce"
)


df_socioeconomico["PIB"] = pd.to_numeric(
    df_socioeconomico["PIB"],
    errors="coerce"
)


df_socioeconomico["Vehiculos_1000_hab"] = (
    df_socioeconomico["Vehiculos_1000_hab"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)

df_socioeconomico["Vehiculos_1000_hab"] = pd.to_numeric(
    df_socioeconomico["Vehiculos_1000_hab"],
    errors="coerce"
)


df_socioeconomico["Años"] = pd.to_numeric(
    df_socioeconomico["Años"],
    errors="coerce"
)


display(
    df_socioeconomico.head()
)

print("\nTipos después de la conversión:")

display(
    df_socioeconomico[
        [
            "Años",
            "Población",
            "PIB",
            "Vehiculos_1000_hab"
        ]
    ].dtypes.to_frame("tipo")
)

,Ciudad,Años,Población,PIB,Vehiculos_1000_hab
0,A Coruña,2013,245923,20600,352.61
1,A Coruña,2014,244810,20600,354.22
2,A Coruña,2015,243870,21800,355.33
3,A Coruña,2016,243978,22700,357.16
4,A Coruña,2017,244099,23300,332.69



Tipos después de la conversión:


,tipo
Años,int64
Población,int64
PIB,int64
Vehiculos_1000_hab,float64


## Comprobación de valores ausentes y cobertura temporal

Antes de agregar las observaciones anuales se comprueba la disponibilidad de información para cada ciudad.

Se analiza:

- el primer año disponible;
- el último año disponible;
- el número de años distintos;
- el número total de registros.

Esta comprobación permite verificar que las ciudades presentan una cobertura temporal comparable y detectar posibles problemas antes de resumir las series.

In [219]:
# =============================================================================
# 7. COMPROBACIÓN DE COBERTURA TEMPORAL
# =============================================================================

resumen_temporal = (
    df_socioeconomico
    .groupby("Ciudad")
    .agg(
        año_inicial=("Años", "min"),
        año_final=("Años", "max"),
        numero_años=("Años", "nunique"),
        numero_registros=("Años", "size")
    )
    .reset_index()
)

display(
    resumen_temporal
)


print("\nValores ausentes por variable:")

display(
    df_socioeconomico[
        [
            "Ciudad",
            "Años",
            "Población",
            "PIB",
            "Vehiculos_1000_hab"
        ]
    ]
    .isna()
    .sum()
    .to_frame("Valores ausentes")
)

,Ciudad,año_inicial,año_final,numero_años,numero_registros
0,A Coruña,2013,2024,12,12
1,Albacete,2013,2024,12,12
2,Alicante,2013,2024,12,12
3,Barcelona,2013,2024,12,12
4,Bilbao,2013,2024,12,12
5,Madrid,2013,2024,12,12
6,Murcia,2013,2024,12,12
7,Santa Cruz de Tenerife,2013,2024,12,12
8,Sevilla,2013,2024,12,12
9,Valencia,2013,2024,12,12



Valores ausentes por variable:


,Valores ausentes
Ciudad,0
Años,0
Población,0
PIB,0
Vehiculos_1000_hab,0


## Agregación de los registros socioeconómicos

El *dataset* socioeconómico contiene una observación por ciudad y año, mientras que el dataset de ciudades contiene una única observación por ciudad.

Para poder realizar una unión uno a uno, es necesario transformar previamente las series anuales en características representativas de cada ciudad.

Se utiliza la mediana temporal de:

- población;
- PIB;
- vehículos por cada 1000 habitantes.

La mediana resume el nivel característico de cada variable durante el período analizado y presenta una mayor robustez frente a valores extremos o posibles discontinuidades puntuales que la media aritmética.

El resultado contiene exactamente una observación socioeconómica por ciudad.

In [220]:
# =============================================================================
# 8. AGREGACIÓN DE LOS DATOS SOCIOECONÓMICOS POR CIUDAD
# =============================================================================

df_socio_ciudad = (
    df_socioeconomico
    .groupby(
        "Ciudad",
        as_index=False
    )
    .agg(
        poblacion_mediana=(
            "Población",
            "median"
        ),

        pib_mediano=(
            "PIB",
            "median"
        ),

        vehiculos_1000_hab_mediana=(
            "Vehiculos_1000_hab",
            "median"
        )
    )
)


print(
    f"Número de ciudades después de la agregación: "
    f"{df_socio_ciudad.shape[0]}"
)

display(
    df_socio_ciudad
)

Número de ciudades después de la agregación: 12


,Ciudad,poblacion_mediana,pib_mediano,vehiculos_1000_hab_mediana
0,A Coruña,245159.0,23800.0,336.945
1,Albacete,172769.0,20050.0,433.050
2,Alicante,334969.5,19550.0,439.805
3,Barcelona,3657085.0,40400.0,358.775
4,Bilbao,790310.0,33650.0,400.760
5,Madrid,4929861.5,45100.0,437.155
6,Murcia,320220.0,19250.0,286.335
7,Santa Cruz de Tenerife,362610.0,20550.0,515.360
8,Sevilla,879838.0,20100.0,453.715
9,Valencia,1398426.5,33300.0,541.760


In [221]:
# =============================================================================
# 9. NORMALIZACIÓN DEL IDENTIFICADOR DE CIUDAD
# =============================================================================

# Detectamos la columna identificadora del dataset de ciudades.
posibles_columnas_ciudad = [
    "city",
    "Ciudad",
    "ciudad",
    "Municipio",
    "municipio",
    "nombre",
    "Nombre"
]

col_ciudad = None

for col in posibles_columnas_ciudad:

    if col in df_ciudades.columns:
        col_ciudad = col
        break


if col_ciudad is None:

    raise ValueError(
        "No se ha encontrado la columna identificadora "
        "de ciudad en el dataset de ciudades."
    )


print(
    f"Columna de ciudad detectada en ciudades.csv: "
    f"{col_ciudad}"
)


# -----------------------------------------------------------------------------
# Creación del identificador normalizado
# -----------------------------------------------------------------------------

df_ciudades["ciudad_normalizada"] = (
    df_ciudades[col_ciudad]
    .apply(normalizar_texto)
)


df_socio_ciudad["ciudad_normalizada"] = (
    df_socio_ciudad["Ciudad"]
    .apply(normalizar_texto)
)


print("\nCiudades del dataset principal:")

display(
    df_ciudades[
        [
            col_ciudad,
            "ciudad_normalizada"
        ]
    ]
)


print("\nCiudades del dataset socioeconómico:")

display(
    df_socio_ciudad[
        [
            "Ciudad",
            "ciudad_normalizada"
        ]
    ]
)

Columna de ciudad detectada en ciudades.csv: Ciudad

Ciudades del dataset principal:


,Ciudad,ciudad_normalizada
0,Madrid,madrid
1,Barcelona,barcelona
2,Bilbao,bilbao
3,Sevilla,sevilla
4,Valencia,valencia
5,Valladolid,valladolid
6,Murcia,murcia
7,Santa Cruz de Tenerife,santa_cruz_de_tenerife
8,A Coruña,a_coruna
9,Zaragoza,zaragoza



Ciudades del dataset socioeconómico:


,Ciudad,ciudad_normalizada
0,A Coruña,a_coruna
1,Albacete,albacete
2,Alicante,alicante
3,Barcelona,barcelona
4,Bilbao,bilbao
5,Madrid,madrid
6,Murcia,murcia
7,Santa Cruz de Tenerife,santa_cruz_de_tenerife
8,Sevilla,sevilla
9,Valencia,valencia


## Comprobación previa de correspondencias

Antes de realizar la unión definitiva, se comprueba qué ciudades aparecen en uno de los conjuntos pero no en el otro. Esta validación es importante porque una unión aparentemente correcta podría generar valores ausentes si alguna ciudad presenta un nombre diferente o no está disponible en una de las fuentes. Idealmente, ambas comprobaciones deben devolver conjuntos vacíos.

In [222]:
# =============================================================================
# 10. COMPROBACIÓN DE CORRESPONDENCIAS ENTRE CIUDADES
# =============================================================================

ciudades_principal = set(
    df_ciudades["ciudad_normalizada"]
)

ciudades_socio = set(
    df_socio_ciudad["ciudad_normalizada"]
)


solo_ciudades = (
    ciudades_principal
    - ciudades_socio
)

solo_socio = (
    ciudades_socio
    - ciudades_principal
)


print(
    "Ciudades presentes en ciudades.csv "
    "pero no en el socioeconómico:"
)

print(
    solo_ciudades
    if solo_ciudades
    else "Ninguna"
)


print(
    "\nCiudades presentes en el socioeconómico "
    "pero no en ciudades.csv:"
)

print(
    solo_socio
    if solo_socio
    else "Ninguna"
)

Ciudades presentes en ciudades.csv pero no en el socioeconómico:
{'alicante_alacant'}

Ciudades presentes en el socioeconómico pero no en ciudades.csv:
{'alicante'}


In [223]:
# =============================================================================
# HOMOGENEIZACIÓN DE CASOS ESPECÍFICOS
# =============================================================================

# Alicante aparece con distinta denominación en ambos datasets:
# "Alicante/Alacant" en ciudades.csv y "Alicante" en el socioeconómico.
#
# Se unifica el identificador normalizado para garantizar la correspondencia
# correcta durante la unión.

df_ciudades["ciudad_normalizada"] = (
    df_ciudades["ciudad_normalizada"]
    .replace({
        "alicante_alacant": "alicante"
    })
)

In [224]:
# =============================================================================
# 10. COMPROBACIÓN DE CORRESPONDENCIAS ENTRE CIUDADES
# =============================================================================

ciudades_principal = set(
    df_ciudades["ciudad_normalizada"]
)

ciudades_socio = set(
    df_socio_ciudad["ciudad_normalizada"]
)


solo_ciudades = (
    ciudades_principal
    - ciudades_socio
)

solo_socio = (
    ciudades_socio
    - ciudades_principal
)


print(
    "Ciudades presentes en ciudades.csv "
    "pero no en el socioeconómico:"
)

print(
    solo_ciudades
    if solo_ciudades
    else "Ninguna"
)


print(
    "\nCiudades presentes en el socioeconómico "
    "pero no en ciudades.csv:"
)

print(
    solo_socio
    if solo_socio
    else "Ninguna"
)

Ciudades presentes en ciudades.csv pero no en el socioeconómico:
Ninguna

Ciudades presentes en el socioeconómico pero no en ciudades.csv:
Ninguna


## Unión de los dos conjuntos

Una vez homogeneizados los identificadores, se realiza una unión de tipo `left join`, tomando como conjunto principal el *dataset* de ciudades. De esta forma se mantienen exactamente las ciudades que forman parte del análisis original y se incorporan sus características socioeconómicas cuando existe correspondencia.

In [225]:
# =============================================================================
# 11. UNIÓN DEL DATASET DE CIUDADES Y EL SOCIOECONÓMICO
# =============================================================================

columnas_socioeconomicas = [
    "ciudad_normalizada",
    "poblacion_mediana",
    "pib_mediano",
    "vehiculos_1000_hab_mediana"
]


df_ciudades_socio = df_ciudades.merge(
    df_socio_ciudad[columnas_socioeconomicas],

    on="ciudad_normalizada",

    how="left",

    validate="one_to_one"
)


print(
    "Dimensiones del dataset resultante:"
)

print(
    f"{df_ciudades_socio.shape[0]} filas x "
    f"{df_ciudades_socio.shape[1]} columnas"
)


display(
    df_ciudades_socio
)

Dimensiones del dataset resultante:
12 filas x 11 columnas


,Ciudad,Latitud,Longitud,Altitud (m),Superficie (km²),Distancia_Mar (km),TPI_Factor_Cuenca,ciudad_normalizada,poblacion_mediana,pib_mediano,vehiculos_1000_hab_mediana
0,Madrid,4.041.694,-370.33300,660.0,604.45,327.66,14.10,madrid,4929861.5,45100.0,437.155
1,Barcelona,413.825,217.69400,30.0,101.35,2.60,-48.24,barcelona,3657085.0,40400.0,358.775
2,Bilbao,4.326.306,-2.93500,24.0,41.60,8.75,-150.72,bilbao,790310.0,33650.0,400.760
3,Sevilla,3.738.861,-599.55600,17.0,141.42,53.66,-21.20,sevilla,879838.0,20100.0,453.715
4,Valencia,39.47,-0.37639,23.0,134.65,3.99,-1.72,valencia,1398426.5,33300.0,541.760
5,Valladolid,4.165.198,-472.85600,704.0,197.91,193.42,-50.06,valladolid,299490.0,26650.0,437.405
6,Murcia,3.798.333,-113.02800,56.0,881.86,35.41,-145.54,murcia,320220.0,19250.0,286.335
7,Santa Cruz de Tenerife,2.846.667,-16.25000,34.0,150.56,0.47,-237.02,santa_cruz_de_tenerife,362610.0,20550.0,515.360
8,A Coruña,4.337.386,-840.00300,21.0,37.83,1.06,-5.36,a_coruna,245159.0,23800.0,336.945
9,Zaragoza,41.65,-0.88333,224.0,973.78,169.66,-191.62,zaragoza,674003.5,28100.0,374.205


In [226]:
# =============================================================================
# 13. PREPARACIÓN DEL DATASET FINAL
# =============================================================================

df_ciudades_socio_final = (
    df_ciudades_socio
    .drop(
        columns=[
            "ciudad_normalizada"
        ]
    )
    .copy()
)


display(
    df_ciudades_socio_final
)

,Ciudad,Latitud,Longitud,Altitud (m),Superficie (km²),Distancia_Mar (km),TPI_Factor_Cuenca,poblacion_mediana,pib_mediano,vehiculos_1000_hab_mediana
0,Madrid,4.041.694,-370.33300,660.0,604.45,327.66,14.10,4929861.5,45100.0,437.155
1,Barcelona,413.825,217.69400,30.0,101.35,2.60,-48.24,3657085.0,40400.0,358.775
2,Bilbao,4.326.306,-2.93500,24.0,41.60,8.75,-150.72,790310.0,33650.0,400.760
3,Sevilla,3.738.861,-599.55600,17.0,141.42,53.66,-21.20,879838.0,20100.0,453.715
4,Valencia,39.47,-0.37639,23.0,134.65,3.99,-1.72,1398426.5,33300.0,541.760
5,Valladolid,4.165.198,-472.85600,704.0,197.91,193.42,-50.06,299490.0,26650.0,437.405
6,Murcia,3.798.333,-113.02800,56.0,881.86,35.41,-145.54,320220.0,19250.0,286.335
7,Santa Cruz de Tenerife,2.846.667,-16.25000,34.0,150.56,0.47,-237.02,362610.0,20550.0,515.360
8,A Coruña,4.337.386,-840.00300,21.0,37.83,1.06,-5.36,245159.0,23800.0,336.945
9,Zaragoza,41.65,-0.88333,224.0,973.78,169.66,-191.62,674003.5,28100.0,374.205


In [227]:
# =============================================================================
# 14. GUARDADO DEL DATASET
# =============================================================================

df_ciudades_socio_final.to_csv(
    RUTA_SALIDA,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Dataset guardado correctamente."
)

print(
    "\nRuta:"
)

print(
    RUTA_SALIDA.resolve()
)

print(
    f"\nDimensiones finales: "
    f"{df_ciudades_socio_final.shape[0]} filas x "
    f"{df_ciudades_socio_final.shape[1]} columnas"
)

Dataset guardado correctamente.

Ruta:
C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\TFM_github\datasets\archivo_ciudades_socioeconomicos\ciudades_socioeconomicos.csv

Dimensiones finales: 12 filas x 10 columnas
